In [1]:
from pathlib import Path
import pandas as pd
import logging
from src.signals_scaling import (compute_moment_scaling_acc,compute_moment_scaling_vel, compute_moment_scaling_disp,
                         compute_scaling_exponents,test_scaling_linearity, fit_piecewise_scaling,
                         trim_to_event_window, compute_increment_tail_exponents, check_increments_distribution)
from src.plot_settings import set_plot_style
from src.plots import (plot_onset_diagnostic, plot_onset_distribution,
                       plot_increments_distribution)
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

INFO | Environment ready


In [2]:
# ============================================================================
# Creazione di df_acc_long COMPLETAMENTE RAW (nessuna preprocessing)
# ============================================================================

import pandas as pd
from pathlib import Path

# 1. Carica i dati raw
print("Loading raw data...")
from src.io import build_accelerations
df_acc_raw = build_accelerations('../data/raw/query.zip')
print(f"Raw data loaded: {df_acc_raw['file'].nunique()} files, {len(df_acc_raw):,} samples")

# 2. Filtra solo i segnali lunghi (>= 48000 samples) - NESSUN'ALTRA MODIFICA
print("\nFiltering long signals...")
min_samples = 48000
signal_lengths = df_acc_raw.groupby('file')['sample'].max() + 1
long_files = signal_lengths[signal_lengths >= min_samples].index
df_acc_long = df_acc_raw[df_acc_raw['file'].isin(long_files)].copy()

print(f"Long signals: {df_acc_long['file'].nunique()} files")
print(f"Excluded: {df_acc_raw['file'].nunique() - df_acc_long['file'].nunique()} short files")

# 3. Verifica - dati completamente RAW
print("\n" + "="*70)
print("VERIFICATION - COMPLETELY RAW DATA:")
print("="*70)
mean_check = df_acc_long.groupby('file')['acceleration'].mean()
std_check = df_acc_long.groupby('file')['acceleration'].std()

print(f"\nMean per file (can be any value, NOT zero):")
print(f"  - Min mean:  {mean_check.min():.4f}")
print(f"  - Max mean:  {mean_check.max():.4f}")
print(f"  - Mean of means: {mean_check.mean():.4f}")

print(f"\nStd per file (should be >> 1 for raw data):")
print(f"  - Min std:  {std_check.min():.2f}")
print(f"  - Max std:  {std_check.max():.2f}")
print(f"  - Mean std: {std_check.mean():.2f}")

print(f"\nAcceleration range:")
print(f"  - Min: {df_acc_long['acceleration'].min():.2f} cm/s²")
print(f"  - Max: {df_acc_long['acceleration'].max():.2f} cm/s²")

print("\n✅ df_acc_long ready: COMPLETELY RAW (no baseline correction, no normalization)")
print(f"   {len(df_acc_long):,} samples, {df_acc_long['file'].nunique()} files")
print(f"   Columns: {df_acc_long.columns.tolist()}")
print("="*70)

Loading raw data...
Raw data loaded: 66 files, 2,614,815 samples

Filtering long signals...
Long signals: 48 files
Excluded: 18 short files

VERIFICATION - COMPLETELY RAW DATA:

Mean per file (can be any value, NOT zero):
  - Min mean:  -0.0000
  - Max mean:  0.0000
  - Mean of means: 0.0000

Std per file (should be >> 1 for raw data):
  - Min std:  0.01
  - Max std:  0.04
  - Mean std: 0.02

Acceleration range:
  - Min: -0.98 cm/s²
  - Max: 0.73 cm/s²

✅ df_acc_long ready: COMPLETELY RAW (no baseline correction, no normalization)
   2,328,000 samples, 48 files
   Columns: ['file', 'sample', 'acceleration']


In [3]:
######### VERSIONE SENZA NORMALIZZAZIONE
logger.info("Loading preprocessed data...")
df_acc_clean = pd.read_parquet('../data/processed/acc_preprocessed_all.parquet')
df_acc_long  = df_acc_long
check(len(df_acc_clean) > 0, f"All signals loaded: {df_acc_clean['file'].nunique()} files")
check(len(df_acc_long) > 0,  f"Long signals loaded: {df_acc_long['file'].nunique()} files")

FIGURES_DIR = Path('../figures/03_single_signal')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / 'scaling' / 'displacement' / 'event_window').mkdir(parents=True, exist_ok=True)
check(FIGURES_DIR.exists(), f"Figures directory ready: {FIGURES_DIR}")

INFO | Loading preprocessed data...
INFO | All signals loaded: 66 files
INFO | Long signals loaded: 48 files
INFO | Figures directory ready: ../figures/03_single_signal


In [5]:
def plot_empirical_pdfs_individual_dual_view(df_acc, bins=100, output_dir='../figures/acceleration_pdfs_individual'):
    """
    Plot empirical PDFs for each file individually with dual view.
    For each file, creates one PDF with 2 subplots:
      - Left: full range of acceleration values
      - Right: zoomed view (x-axis limited to [-4, 4])
    
    Parameters
    ----------
    df_acc : pd.DataFrame
        DataFrame with columns [file, sample, acceleration]
    bins : int
        Number of histogram bins
    output_dir : str
        Directory to save figures
    
    Returns
    -------
    None (saves individual PDF files to disk)
    """
    from pathlib import Path
    import matplotlib.pyplot as plt
    import numpy as np
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    files = df_acc['file'].unique()
    n_files = len(files)
    
    print(f"Plotting dual-view PDFs for {n_files} files...")
    print("Each plot has 2 subplots: full range (left) + zoomed [-4,4] (right)\n")
    
    for idx, file in enumerate(files):
        # Get acceleration data for this file
        data = df_acc[df_acc['file'] == file]['acceleration'].values
        
        # Extract station info
        station = file.split('.')[1]
        stream = file.split('.')[3] if len(file.split('.')) > 3 else 'UNK'
        
        # Statistics
        mean_val = np.mean(data)
        std_val = np.std(data)
        min_val = np.min(data)
        max_val = np.max(data)
        abs_max = np.max(np.abs(data))
        n_samples = len(data)
        
        # Percentages
        pct_less_1 = 100 * np.sum(np.abs(data) < 1.0) / len(data)
        pct_less_05 = 100 * np.sum(np.abs(data) < 0.5) / len(data)
        n_zeros = np.sum(data == 0)
        pct_zeros = 100 * n_zeros / len(data)
        
        # Create figure with 2 subplots side by side
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # --- SUBPLOT 1: FULL RANGE ---
        axes[0].hist(data, bins=bins, edgecolor='black', linewidth=0.5, 
                    alpha=0.7, density=True, color='steelblue')
        
        # Reference lines
        axes[0].axvline(0, color='red', linewidth=2, linestyle='--', 
                       alpha=0.7, label='Zero', zorder=10)
        axes[0].axvline(-1, color='orange', linewidth=1.5, linestyle=':', 
                       alpha=0.7, label='|a|=1', zorder=10)
        axes[0].axvline(1, color='orange', linewidth=1.5, linestyle=':', 
                       alpha=0.7, zorder=10)
        
        # Labels
        axes[0].set_xlabel('Acceleration a(t) [cm/s²]', fontsize=13)
        axes[0].set_ylabel('Probability density', fontsize=13)
        axes[0].set_title(f'Full range: [{min_val:.2f}, {max_val:.2f}] cm/s²', fontsize=12)
        axes[0].legend(fontsize=11)
        axes[0].grid(True, alpha=0.3, axis='y')
        
        # --- SUBPLOT 2: ZOOMED TO [-4, 4] ---
        axes[1].hist(data, bins=bins, edgecolor='black', linewidth=0.5, 
                    alpha=0.7, density=True, color='steelblue')
        
        # Reference lines
        axes[1].axvline(0, color='red', linewidth=2, linestyle='--', 
                       alpha=0.7, label='Zero', zorder=10)
        axes[1].axvline(-1, color='orange', linewidth=1.5, linestyle=':', 
                       alpha=0.7, label='|a|=1', zorder=10)
        axes[1].axvline(1, color='orange', linewidth=1.5, linestyle=':', 
                       alpha=0.7, zorder=10)
        
        # ZOOM TO [-4, 4]
        axes[1].set_xlim(-4, 4)
        
        # Labels
        axes[1].set_xlabel('Acceleration a(t) [cm/s²]', fontsize=13)
        axes[1].set_ylabel('Probability density', fontsize=13)
        axes[1].set_title('Zoomed view: [-4, 4] cm/s²', fontsize=12)
        axes[1].legend(fontsize=11)
        axes[1].grid(True, alpha=0.3, axis='y')
        
        # --- MAIN TITLE ---
        main_title = (f'Empirical PDF — Station: {station}, Stream: {stream}\n'
                     f'μ = {mean_val:.4f}, σ = {std_val:.4f}, max|a| = {abs_max:.3f} cm/s² | '
                     f'N = {n_samples:,} samples\n'
                     f'%|a|<1: {pct_less_1:.1f}%, %|a|<0.5: {pct_less_05:.1f}%, %a=0: {pct_zeros:.1f}%')
        fig.suptitle(main_title, fontsize=12, y=0.98)
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        
        # Save
        filename = f'pdf_{station}_{stream}.pdf'
        filepath = Path(output_dir) / filename
        plt.savefig(filepath, bbox_inches='tight', dpi=150)
        plt.close()
        
        if (idx + 1) % 10 == 0 or (idx + 1) == n_files:
            print(f"  Progress: {idx+1}/{n_files} dual-view PDFs saved")
    
    print(f"\n✅ All {n_files} dual-view PDFs saved to: {output_dir}/")
    print(f"   Each PDF contains 2 subplots: full range + zoomed [-4,4]")


# USO:
plot_empirical_pdfs_individual_dual_view(
    df_acc_long, 
    bins=100,
    output_dir= FIGURES_DIR / 'acceleration_pdfs_individual_dual_view'
)

Plotting dual-view PDFs for 48 files...
Each plot has 2 subplots: full range (left) + zoomed [-4,4] (right)

  Progress: 10/48 dual-view PDFs saved
  Progress: 20/48 dual-view PDFs saved
  Progress: 30/48 dual-view PDFs saved
  Progress: 40/48 dual-view PDFs saved
  Progress: 48/48 dual-view PDFs saved

✅ All 48 dual-view PDFs saved to: ../figures/03_single_signal/acceleration_pdfs_individual_dual_view/
   Each PDF contains 2 subplots: full range + zoomed [-4,4]
